<a href="https://colab.research.google.com/github/Avichatt/ML-projects/blob/main/TATA-STeel_Defect%20detection%20in%20hot%20rolling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing libraries and dependencies

#

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

Loading data from the csv files

In [3]:
train_df = pd.read_csv('/content/sample_data/train.csv')
test_df  = pd.read_csv('/content/sample_data/test.csv')
sample_sub = pd.read_csv('/content/sample_data/sample_submission.csv')

print("=" * 60)
print("STEP 2 - Data Loaded Successfully")
print("=" * 60)
print(f"Training data shape  : {train_df.shape}")
print(f"Test data shape      : {test_df.shape}")
print(f"Submission shape     : {sample_sub.shape}")

STEP 2 - Data Loaded Successfully
Training data shape  : (1352, 51)
Test data shape      : (339, 50)
Submission shape     : (10, 2)


Class Distribution and imbalance ratio






In [14]:
print("=" * 60)
print("Target Variable Distribution")
print("=" * 60)
print(train_df['Y'].value_counts())

defect_count    = train_df['Y'].sum()
no_defect_count = len(train_df) - defect_count
imbalance_ratio = no_defect_count / defect_count

print(f"\nDefective coils (Y=1)     : {int(defect_count)}")
print(f"Non-defective coils (Y=0) : {int(no_defect_count)}")
print(f"Imbalance ratio           : {imbalance_ratio:.1f} : 1")


# This is an IMBALANCED dataset — very few defects compared to non-defects.
# That's why normal accuracy isn't enough; we need to focus on Recall & Precision.


Target Variable Distribution
Y
0.0    1286
1.0      66
Name: count, dtype: int64

Defective coils (Y=1)     : 66
Non-defective coils (Y=0) : 1286
Imbalance ratio           : 19.5 : 1


Feature Preparation

In [15]:
feature_columns = [col for col in train_df.columns if col not in ['CoilID', 'Y']]

X_train = train_df[feature_columns].copy()
y_train = train_df['Y'].copy()

X_test  = test_df[feature_columns].copy()

print("=" * 60)
print("Features Prepared")
print("=" * 60)
print(f"Number of features: {len(feature_columns)}")
print(f"Features: {feature_columns[:5]} ... {feature_columns[-5:]}")

Features Prepared
Number of features: 49
Features: ['X1', 'X2', 'X3', 'X4', 'X5'] ... ['X45', 'X46', 'X47', 'X48', 'X49']


handle Missing values

In [16]:
missing_before = X_train.isnull().sum().sum()
print("=" * 60)
print("Handling Missing Values")
print("=" * 60)
print(f"Missing values in training data BEFORE: {missing_before}")

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test  = X_test.fillna(medians)

missing_after = X_train.isnull().sum().sum()
print(f"Missing values in training data AFTER : {missing_after}")


# Some cells in the CSV are empty (missing data).
# We fill them with the MEDIAN of that column.
# Median is better than mean because it's not affected by extreme values (outliers).

Handling Missing Values
Missing values in training data BEFORE: 249
Missing values in training data AFTER : 0


Feature scaling

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns)
X_test_scaled  = pd.DataFrame(X_test_scaled, columns=feature_columns)

print("=" * 60)
print("Feature Scaling Complete")
print("=" * 60)
print("All features scaled (mean ~ 0, std ~ 1)")
print(f"Example - X1 mean: {X_train_scaled['X1'].mean():.4f}, std: {X_train_scaled['X1'].std():.4f}")

# Different features have very different ranges (e.g., X1 might be 300–1100, X11 might be 24–39).
# StandardScaler converts all features to the same scale: mean=0, standard deviation=1.
# This helps the model treat all features fairly.

STEP 6 - Feature Scaling Complete
All features scaled (mean ~ 0, std ~ 1)
Example - X1 mean: -0.0000, std: 1.0004


Build Model

In [17]:
print("=" * 60)
print("Building XGBoost Model")
print("=" * 60)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=imbalance_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

print("Model configuration completed.")


# XGBoost (eXtreme Gradient Boosting) is one of the best ML algorithms for tabular data.
# It builds many small decision trees one after another, each fixing the mistakes of the previous one.
#
# KEY PARAMETER: scale_pos_weight
#   Since we have far more non-defects than defects, we tell the model to pay MORE attention
#   to the defective samples. We set it to the imbalance ratio so the model treats both
#   classes equally in importance.
#
# WHY THIS HELPS WITH RECALL:
#   By giving more weight to defective coils, the model becomes very reluctant to miss any defect,
#   which pushes Recall toward 100%.

Building XGBoost Model
Model configuration completed.


5-fold cross validation

In [18]:
print("=" * 60)
print("5-Fold Cross-Validation")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_recalls     = []
fold_precisions  = []

for fold_num, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train), 1):
    X_fold_train, y_fold_train = X_train_scaled.iloc[train_idx], y_train.iloc[train_idx]
    X_fold_val, y_fold_val = X_train_scaled.iloc[val_idx], y_train.iloc[val_idx]

    # Temporarily fit model to fold training partition
    model.fit(X_fold_train, y_fold_train)
    val_probs = model.predict_proba(X_fold_val)[:, 1]

    best_threshold = 0.5
    best_precision = 0.0

    for threshold in np.arange(0.01, 1.0, 0.01):
        y_pred_temp = (val_probs >= threshold).astype(int)
        recall_temp = recall_score(y_fold_val, y_pred_temp, zero_division=0)
        precision_temp = precision_score(y_fold_val, y_pred_temp, zero_division=0)

        if recall_temp == 1.0 and precision_temp > best_precision:
            best_precision = precision_temp
            best_threshold = threshold

    y_fold_pred = (val_probs >= best_threshold).astype(int)
    fold_recall    = recall_score(y_fold_val, y_fold_pred, zero_division=0)
    fold_precision = precision_score(y_fold_val, y_fold_pred, zero_division=0)

    fold_recalls.append(fold_recall)
    fold_precisions.append(fold_precision)
    print(f"  Fold {fold_num}: Threshold={best_threshold:.2f}  Recall={fold_recall:.4f}  Precision={fold_precision:.4f}")

avg_recall    = np.mean(fold_recalls)
avg_precision = np.mean(fold_precisions)
print(f"\n  >>> Average CV Recall    : {avg_recall:.4f}")
print(f"  >>> Average CV Precision : {avg_precision:.4f}")



# Before making final predictions, we test how well the model performs.
# StratifiedKFold splits data into 5 parts, trains on 4, tests on 1, and rotates.
# "Stratified" means each fold keeps the same ratio of defects to non-defects.

5-Fold Cross-Validation
  Fold 1: Threshold=0.50  Recall=0.3077  Precision=0.6667
  Fold 2: Threshold=0.50  Recall=0.2143  Precision=0.3000
  Fold 3: Threshold=0.50  Recall=0.1538  Precision=0.2000
  Fold 4: Threshold=0.50  Recall=0.3077  Precision=0.4444
  Fold 5: Threshold=0.50  Recall=0.4615  Precision=0.6000

  >>> Average CV Recall    : 0.2890
  >>> Average CV Precision : 0.4422


Train Final model and calibrate threshold

In [19]:
print("=" * 60)
print("Training & Calibration")
print("=" * 60)

model.fit(X_train_scaled, y_train)

train_probs = model.predict_proba(X_train_scaled)[:, 1]

best_final_threshold = 0.5
best_final_precision = 0.0

for threshold in np.arange(0.01, 1.0, 0.01):
    y_pred_temp = (train_probs >= threshold).astype(int)
    recall_temp    = recall_score(y_train, y_pred_temp, zero_division=0)
    precision_temp = precision_score(y_train, y_pred_temp, zero_division=0)

    if recall_temp == 1.0 and precision_temp > best_final_precision:
        best_final_precision = precision_temp
        best_final_threshold = threshold

print(f"Best calibrated threshold: {best_final_threshold:.2f}")
print(f"Precision at this threshold: {best_final_precision:.4f}")

# By default, if probability > 0.5, predict "defect". But 0.5 might miss some defects.
# To get 100% recall, we LOWER the threshold: if there's even a small chance of defect,
# we flag it. We find the lowest threshold that catches ALL defects (recall=100%)
# while keeping precision as high as possible.

Training & Calibration
Best calibrated threshold: 0.29
Precision at this threshold: 1.0000


Sanity Check

In [20]:
print("=" * 60)
print("Training Data Sanity Check")
print("=" * 60)

y_train_pred = (train_probs >= best_final_threshold).astype(int)
cm = confusion_matrix(y_train, y_train_pred)

print("\nConfusion Matrix:")
print(f"  True Negatives (Said healthy, was healthy)   : {cm[0][0]}")
print(f"  False Positives (Said defect, was healthy)  : {cm[0][1]}")
print(f"  False Negatives (Missed defect - CRITICAL!) : {cm[1][0]}")
print(f"  True Positives (Caught defect)              : {cm[1][1]}")

train_recall    = recall_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
print(f"\n  Recall    : {train_recall:.4f}  ({'PASS' if train_recall == 1.0 else 'FAIL'})")
print(f"  Precision : {train_precision:.4f}  ({'PASS' if train_precision > 0.9 else 'FAIL'})")

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred, target_names=['No Defect (0)', 'Defect (1)']))

Training Data Sanity Check

Confusion Matrix:
  True Negatives (Said healthy, was healthy)   : 1286
  False Positives (Said defect, was healthy)  : 0
  False Negatives (Missed defect - CRITICAL!) : 0
  True Positives (Caught defect)              : 66

  Recall    : 1.0000  (PASS)
  Precision : 1.0000  (PASS)

Classification Report:
               precision    recall  f1-score   support

No Defect (0)       1.00      1.00      1.00      1286
   Defect (1)       1.00      1.00      1.00        66

     accuracy                           1.00      1352
    macro avg       1.00      1.00      1.00      1352
 weighted avg       1.00      1.00      1.00      1352



Make Test prediction and generate submission file

In [21]:
print("=" * 60)
print("Generating Submission")
print("=" * 60)

test_probs = model.predict_proba(X_test_scaled)[:, 1]
test_predictions = (test_probs >= best_final_threshold).astype(int)

submission = pd.DataFrame({
    'CoilID': test_df['CoilID'],
    'Y': test_predictions
})

submission.to_csv('expected_submission.csv', index=False)
print("Submission saved successfully to 'expected_submission.csv'!")
print(f"Defects Predicted in Test set: {test_predictions.sum()} out of {len(test_predictions)}")
print(submission.head(10))

Generating Submission
Submission saved successfully to 'expected_submission.csv'!
Defects Predicted in Test set: 17 out of 339
   CoilID  Y
0     711  0
1    1542  0
2    1232  0
3     600  0
4    1087  0
5    1401  0
6     217  0
7     877  0
8    1117  0
9     555  0


Feature Importance

In [13]:
print("=" * 60)
print("STEP 14 - Top 10 Most Important Features")
print("=" * 60)

importance = model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': importance
}).sort_values('Importance', ascending=False)

for i, row in feat_imp.head(10).iterrows():
    bar = '#' * int(row['Importance'] * 100)
    print(f"  {row['Feature']:>4s} : {row['Importance']:.4f}  {bar}")

STEP 14 - Top 10 Most Important Features
   X35 : 0.0948  #########
   X36 : 0.0744  #######
   X13 : 0.0711  #######
   X31 : 0.0542  #####
   X32 : 0.0491  ####
   X34 : 0.0435  ####
   X11 : 0.0316  ###
   X49 : 0.0251  ##
   X39 : 0.0250  ##
    X7 : 0.0248  ##
